In [1]:
from pathlib import Path

import numpy as np
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import zero_one_loss
from sklearn.model_selection import train_test_split
from threadpoolctl import threadpool_limits


DATA_DIR = Path.cwd()
if (DATA_DIR / "homework_3").is_dir():
    DATA_DIR = DATA_DIR / "homework_3"

SEED = 42
BOOSTING_TREES = (1, 10, 100, 300, 1000)


def run_boosting(path):
    rows = np.loadtxt(path, delimiter=",")
    x, y = rows[:, 1:], rows[:, 0].astype(int)
    x_train, x_test, y_train, y_test = train_test_split(
        x, y, test_size=0.2, random_state=SEED
    )
    scores = []
    for trees in BOOSTING_TREES:
        model = GradientBoostingClassifier(
            n_estimators=trees, learning_rate=0.1, max_depth=3, random_state=SEED
        ).fit(x_train, y_train)
        scores.append({
            "trees": trees,
            "train_error": zero_one_loss(y_train, model.predict(x_train)),
            "test_error": zero_one_loss(y_test, model.predict(x_test)),
        })
    order = np.argsort(x[:, 0], kind="stable")
    return {
        "file": path.name,
        "sizes": {"train": len(y_train), "test": len(y_test)},
        "scores": scores,
        "first_feature_positive_rates": [
            float(y[group].mean()) for group in np.array_split(order, 5)
        ],
    }

In [2]:
with threadpool_limits(limits=1):
    datasets = [run_boosting(DATA_DIR / name) for name in ("dset0-1.csv", "dset1-1.csv")]

for dataset in datasets:
    print(f"{dataset['file']}: {dataset['sizes']['train']} train, {dataset['sizes']['test']} test")
print(f"{'Trees':>6} {'dset0 train':>13} {'dset0 test':>12} {'dset1 train':>13} {'dset1 test':>12}")
for a, b in zip(datasets[0]["scores"], datasets[1]["scores"], strict=True):
    print(
        f"{a['trees']:>6,} {a['train_error']:>13.2%} {a['test_error']:>12.2%}"
        f" {b['train_error']:>13.2%} {b['test_error']:>12.2%}"
    )

dset0-1.csv: 400 train, 100 test
dset1-1.csv: 400 train, 100 test
 Trees   dset0 train   dset0 test   dset1 train   dset1 test
     1        42.50%       43.00%        43.75%       45.00%
    10        13.25%       27.00%        12.75%       23.00%
   100         0.50%       18.00%         1.25%       20.00%
   300         0.00%       17.00%         0.00%       18.00%
 1,000         0.00%       19.00%         0.00%       18.00%


In [3]:
for dataset in datasets:
    rates = ", ".join(f"{rate:.0%}" for rate in dataset["first_feature_positive_rates"])
    print(f"{dataset['file']}: {rates}")

dset0-1.csv: 52%, 37%, 32%, 34%, 58%
dset1-1.csv: 89%, 63%, 47%, 20%, 8%
